# Case study: checking a vendor extract

This synthetic vendor extract contains two re-sent orders that repeat
the internal key and create exact duplicate rows. It also has a nearly
unique external reference, invoice numbers in two formats, and a missing
business day.

In [ ]:
import datetime as dt
import random

import polars as pl

import onceover

rng = random.Random(2024)
n = 4_000

order_id = list(range(n))
external_ref = [f"ORD-{i:06}" for i in range(n)]
for k in range(3):                       # three duplicated references
    external_ref[n - 1 - k] = external_ref[k]
region_id = [rng.randrange(3) for _ in range(n)]
region_name = [["emea", "amer", "apac"][r] for r in region_id]
invoice_no = [f"INV-{i:05}" if i % 2 == 0 else f"{i:05}" for i in range(n)]
quantity = [rng.randrange(1, 11) for _ in range(n)]
unit_price = [round(rng.uniform(5, 25), 2) for _ in range(n)]
line_total = [round(q * p, 2) for q, p in zip(quantity, unit_price)]

base = dt.date(2024, 3, 4)
skip = {dt.date(2024, 4, 2), dt.date(2024, 4, 3)}     # two weekdays missing
business, i = [], 0
while len(business) < n:
    d = base + dt.timedelta(days=i // 20)
    if d not in skip:
        business.append(d)
    i += 1

df = pl.DataFrame({
    "order_id": order_id,
    "external_ref": external_ref,
    "region_id": region_id,
    "region_name": region_name,
    "invoice_no": invoice_no,
    "quantity": quantity,
    "unit_price": unit_price,
    "line_total": line_total,
    "business_date": business,
})
df = pl.concat([df, df[500:502]])        # two duplicated rows, on purpose

report = onceover.profile(df)
print(report.overview["headline"])

The headline reports no key candidate. The two repeated orders mean no
single column is unique across all rows. Review the ranked findings to
see which issues affect joins, date checks, and invoice-number parsing.

In [ ]:
for f in report.findings:
    kind = "/".join(p for p in (f["category"], f.get("type")) if p)
    print(f"[{f['interestingness']:.2f} {kind}] {f['claim']}")

**Review exact duplicates before using the key.** `order_id` is flagged nearly
unique, with two repeated values. The duplicates finding reports two
rows that repeat earlier rows across every compared column. Its
specification records exact equality with missing values comparing equal. In this fixture the repeated rows
are planted re-sends. In a real extract, confirm whether repeated rows are
redundant before removing them; equality alone does not establish why
they occurred.

**The external reference is not a key.** `external_ref` is nearly unique
with *five* repeated values, and the duplicates finding does not explain
all of them: some rows with repeated references differ. Before using
`external_ref` as a join key, confirm that it is intended to be unique
and investigate the repeated references.

In [ ]:
dirty = next(f for f in report.findings
              if f["category"] == "uniqueness" and "external_ref" in f["subject"]["columns"])
for e in dirty["evidence"]:
    if "examples" in e["metrics"]:
        print("repeated external_ref values:", e["metrics"]["examples"])

In [ ]:
decisions = pl.DataFrame({
    "finding": [
        "order_id nearly unique; 2 rows repeat earlier values exactly",
        "external_ref nearly unique (5 repeats, rows differ)",
        "invoice_no mixes 2 formats",
        "2 business dates missing",
        "line_total predictable from quantity × unit_price",
    ],
    "decision": [
        "deduplicate on order_id; then load key",
        "do not join on it; query vendor",
        "normalize in staging before parsing",
        "send dates back to vendor",
        "keep as recomputation check column",
    ],
    "basis": ["measured · full", "measured · full",
              "measured · full", "measured · full", "estimated · holdout"],
})

## Other relationships and coverage

There is no functional dependency from `quantity` alone to `line_total`
in this fixture: the total also depends on `unit_price`. The region
lookup is verified in both directions with zero violations. That
dependency is expected structure, rather than a data-quality error.

Coverage records which passes ran and their limits. FD verification
uses the full 4,002-row table. Model passes use the encoded rows with
their stated fitting limits and validation partitions
([](../usage/coverage.ipynb)).

Use the counts and comparisons to identify records to review. Confirm
the intended keys, formats, and calendar with the data owner before
changing the extract.